In [23]:
import numpy as np
import jax
import jax.numpy as jnp
from safetensors import safe_open
from safetensors.numpy import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download

In [24]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [25]:
class GELU:
    def __call__(self, x):
        return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

    def __str__(self):
        return "GELU"

In [26]:
class Linear:
    def __init__(self, weights, bias) -> None:
        self.weights = weights
        self.bias = bias

    def __call__(self, x):
        return x @ self.weights + self.bias

    def __str__(self):
        return f"Linear weights: {self.weights.shape}, bias: {self.bias.shape}"

In [27]:
class LayerNorm:
    def __init__(self, weight, bias, eps=1e-5):
        self.weight = weight
        self.bias = bias
        self.eps = eps

    def __call__(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)

        x_norm = (x - mean) / np.sqrt(var + self.eps)

        return self.weight * x_norm + self.bias

    def __str__(self):
        return f"LayerNorm weights: {self.weight.shape}, bias: {self.bias.shape}"

In [28]:
class Attention:
    def __init__(self, attn_weights, attn_bias, proj_weights, proj_bias) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias

    def __call__(self, x):
        qkv = x @ self.attn_weights + self.attn_bias
        q, k, v = np.split(qkv, 3, axis=-1)
        scores = q @ k.T
        scores = scores / np.sqrt(k.shape[-1])
        scores = softmax(scores)
        out = scores @ v
        out = out @ self.proj_weights + self.proj_bias
        return out

    def __str__(self):
        return f"Attention weights: {self.attn_weights.shape}, bias: {self.attn_bias.shape}, proj weights: {self.proj_weights.shape}, proj bias: {self.proj_bias.shape}"

In [29]:
class MultiHeadAttention:
    def __init__(
        self, attn_weights, attn_bias, proj_weights, proj_bias, num_heads=8
    ) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias
        self.num_heads = num_heads

    def __call__(self, x):
        seq_len = x.shape[0]
        hidden_dim = x.shape[1]  # Should be 512
        head_dim = hidden_dim // self.num_heads  # 512 // 8 = 64

        # 1. Project input to Q, K, V
        qkv = x @ self.attn_weights + self.attn_bias  # (seq_len, 512*3)
        q, k, v = np.split(qkv, 3, axis=-1)  # Each: (seq_len, 512)

        # 2. Split into multiple heads
        # Reshape from (seq_len, 512) to (seq_len, num_heads, head_dim)
        q = q.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        k = k.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        v = v.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)

        # 3. Transpose to (num_heads, seq_len, head_dim) for batch processing
        q = q.transpose(1, 0, 2)  # (8, seq_len, 64)
        k = k.transpose(1, 0, 2)  # (8, seq_len, 64)
        v = v.transpose(1, 0, 2)  # (8, seq_len, 64)

        # 4. Compute attention scores for all heads
        scores = q @ k.transpose(0, 2, 1)  # (8, seq_len, seq_len)
        scores = scores / np.sqrt(head_dim)  # Scale by sqrt(64)

        # 5. Apply causal mask (prevent attending to future tokens)
        causal_mask = np.tril(np.ones((seq_len, seq_len)))  # Lower triangular matrix
        scores = np.where(causal_mask == 0, -1e10, scores)  # Mask future positions

        # 6. Apply softmax to get attention weights
        attn_weights = softmax(scores, axis=-1)  # (8, seq_len, seq_len)

        # 7. Apply attention weights to values
        output = attn_weights @ v  # (8, seq_len, 64)

        # 8. Concatenate heads back together
        output = output.transpose(1, 0, 2)  # (seq_len, 8, 64)
        output = output.reshape(seq_len, hidden_dim)  # (seq_len, 512)

        # 9. Final projection
        output = output @ self.proj_weights + self.proj_bias

        return output

    def __str__(self):
        return f"MultiHeadAttention weights: {self.attn_weights.shape}, bias: {self.attn_bias.shape}, proj weights: {self.proj_weights.shape}, proj bias: {self.proj_bias.shape}"

In [30]:
class Transformer:
    def __init__(self, model) -> None:
        self.model = model
        self.attention_blocks = []
        self.ffn_blocks = []

        for i in range(4):
            # Attention block (LayerNorm + Attention)
            ln_1_weight = self.model[f"transformer.h.{i}.ln_1.weight"]
            ln_1_bias = self.model[f"transformer.h.{i}.ln_1.bias"]
            attn_c_attn_weight = self.model[f"transformer.h.{i}.attn.c_attn.weight"]
            attn_c_attn_bias = self.model[f"transformer.h.{i}.attn.c_attn.bias"]
            attn_c_proj_weight = self.model[f"transformer.h.{i}.attn.c_proj.weight"]
            attn_c_proj_bias = self.model[f"transformer.h.{i}.attn.c_proj.bias"]

            self.attention_blocks.append(
                {
                    "ln": LayerNorm(ln_1_weight, ln_1_bias),
                    "attn": MultiHeadAttention(
                        attn_c_attn_weight,
                        attn_c_attn_bias,
                        attn_c_proj_weight,
                        attn_c_proj_bias,
                    ),
                }
            )

            # FFN block (LayerNorm + Linear + GELU + Linear)
            ln_2_weight = self.model[f"transformer.h.{i}.ln_2.weight"]
            ln_2_bias = self.model[f"transformer.h.{i}.ln_2.bias"]
            ffn_1_weight = self.model[f"transformer.h.{i}.mlp.c_fc.weight"]
            ffn_1_bias = self.model[f"transformer.h.{i}.mlp.c_fc.bias"]
            ffn_2_weight = self.model[f"transformer.h.{i}.mlp.c_proj.weight"]
            ffn_2_bias = self.model[f"transformer.h.{i}.mlp.c_proj.bias"]

            self.ffn_blocks.append(
                {
                    "ln": LayerNorm(ln_2_weight, ln_2_bias),
                    "linear1": Linear(ffn_1_weight, ffn_1_bias),
                    "gelu": GELU(),
                    "linear2": Linear(ffn_2_weight, ffn_2_bias),
                }
            )

    def __call__(self, x):
        for attn_block, ffn_block in zip(self.attention_blocks, self.ffn_blocks):
            # Attention with residual
            attn_out = attn_block["attn"](attn_block["ln"](x))
            x += attn_out  # ← Residual connection

            # FFN with residual
            ffn_out = ffn_block["linear2"](
                ffn_block["gelu"](ffn_block["linear1"](ffn_block["ln"](x)))
            )
            x += ffn_out  # ← Residual connection

        # final layer norm
        x = LayerNorm(
            self.model["transformer.ln_f.weight"], self.model["transformer.ln_f.bias"]
        )(x)

        logits = x @ self.model["transformer.wte.weight"].T
        token_ids = np.argmax(logits, axis=-1)

        return int(token_ids[-1])

In [31]:
from abc import ABC, abstractmethod


class Model(ABC):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    @abstractmethod
    def next_token(self, input_text: str) -> str:
        pass

    def generate(self, input_text: str, max_tokens: int = 100) -> str:
        text = input_text
        eos_token = getattr(self.tokenizer, "eos_token", None)

        for _ in range(max_tokens):
            next_token = self.next_token(text)
            text += next_token
            if eos_token and next_token == eos_token:
                break

        return text

In [32]:
class GPT2(Model):
    def __init__(self) -> None:
        from huggingface_hub import hf_hub_download

        tokenizer = AutoTokenizer.from_pretrained(
            "erwanf/gpt2-mini", cache_dir="./data/hf"
        )
        super().__init__(tokenizer)

        # Download the safetensors file from Hugging Face
        model_file = hf_hub_download(
            repo_id="erwanf/gpt2-mini",
            filename="model.safetensors",
            cache_dir="./data/hf",
        )

        # Load weights directly as NumPy arrays - no PyTorch needed!
        self.model_weights = load_file(model_file)

        self.model = Transformer(self.model_weights)

    def next_token(self, input_text: str) -> str:
        input_tokens = self.tokenizer.encode(input_text)
        token_embeddings = self.model_weights["transformer.wte.weight"][input_tokens]
        pos_embeddings = self.model_weights["transformer.wpe.weight"][
            : len(input_tokens)
        ]
        embeddings = token_embeddings + pos_embeddings
        next_token_id = self.model(embeddings)
        return self.tokenizer.decode(next_token_id)


In [33]:
text = "Anthropic is a company"

gpt2 = GPT2()
result = gpt2.generate(text, max_tokens=50)
print(result)

Anthropic is a company that has been working on the project for over a decade.



quer.





quer.









ument.

ument.

ument.

ument.


In [34]:
class RMSNorm():
    def __init__(self, w, eps=1e-6) -> None:
        self.w = w
        self.eps = eps

    def __call__(self, x):
        rms = jnp.sqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + self.eps)
        return (x / rms) * self.w

In [35]:
class SmolLM2TransformerBlock():
    def __init__(self, weights) -> None:
        self.weights = weights

    def __call__(self, x):
        ans = x.copy()

        layer_1 = [
            RMSNorm(self.weights["input_layernorm"]),
            GroupedQueryAttention(
                self.weights["self_attn_q_proj"],
                self.weights["self_attn_k_proj"],
                self.weights["self_attn_v_proj"],
                self.weights["self_attn_o_proj"],
            ),
        ]
        layer_2 = [
            RMSNorm(x, self.weights["post_attention_layernorm"])
        ]

        layer_result = ans
        for layer in layer_1:
            layer_result = layer(layer_result)
        ans += layer_result

        layer_result = ans.copy()
        for layer in layer_2:
            layer_result = layer(layer_result)
        ans += layer_result

        return ans


In [79]:
class SmolLM2(Model):
    def __init__(self) -> None:
        self.layers = []

        tokenizer = AutoTokenizer.from_pretrained(
            "unsloth/SmolLM2-135M-Instruct",
            cache_dir="./data/hf",
        )
        super().__init__(tokenizer)

        model_file = hf_hub_download(
            repo_id="unsloth/SmolLM2-135M-Instruct",
            filename="model.safetensors",
            cache_dir="./data/hf",
        )

        self.model_weights = {}
        with safe_open(model_file, framework="jax") as f:
            for key in f.keys():
                tensor = f.get_tensor(key)
                if hasattr(tensor, "dtype") and str(tensor.dtype) == "bfloat16":
                    tensor = tensor.astype(np.float32)
                self.model_weights[key] = tensor

        layers_count = 30
        for i in range(layers_count):
            weights = {
                "input_layernorm": self.model_weights[f"model.layers.{i}.input_layernorm.weight"],
                "mlp_down_proj": self.model_weights[f"model.layers.{i}.mlp.down_proj.weight"],
                "mlp_gate_proj": self.model_weights[f"model.layers.{i}.mlp.gate_proj.weight"],
                "mlp_up_proj": self.model_weights[f"model.layers.{i}.mlp.up_proj.weight"],
                "post_attention_layernorm": self.model_weights[f"model.layers.{i}.post_attention_layernorm.weight"],
                "self_attn_k_proj": self.model_weights[f"model.layers.{i}.self_attn.k_proj.weight"],
                "self_attn_o_proj": self.model_weights[f"model.layers.{i}.self_attn.o_proj.weight"],
                "self_attn_q_proj": self.model_weights[f"model.layers.{i}.self_attn.q_proj.weight"],
                "self_attn_v_proj": self.model_weights[f"model.layers.{i}.self_attn.v_proj.weight"],
            }
            self.layers.append(SmolLM2TransformerBlock(weights))

    def next_token(self, input_text: str) -> str:
        input_tokens = jnp.array(self.tokenizer.encode(input_text))
        token_embeddings = self.model_weights["model.embed_tokens.weight"][input_tokens]

        x = token_embeddings
        for layer in self.layers:
            x = layer(x)

        x = RMSNorm(self.model_weights["model.norm.weight"])(x)

        logits = x @ self.model_weights["model.embed_tokens.weight"].T
        token_ids = jnp.argmax(logits[-1, :], axis=-1)

        next_token_id = int(token_ids)
        return self.tokenizer.decode([next_token_id])


In [49]:
class GroupedQueryAttention():
    def __init__(self, wq, wk, wv, wo, q_count=9, kv_count=3) -> None:
        # self.wq = q.reshape(q_count, 576 // q_count, 576).transpose(2, 0, 1)
        # self.wk = k.reshape(kv_count, 192 // kv_count, 576).transpose(2, 0, 1)
        # self.wv = v.reshape(kv_count, 192 // kv_count, 576).transpose(2, 0, 1)
        self.wq = wq
        self.wk = wk.repeat(3, axis=0)
        self.wv = wv.repeat(3, axis=0)
        self.wo = wo

    def __call__(self, x):
        seq_len = x.shape[0]

        q = x @ self.wq
        k = x @ self.wk
        v = x @ self.wv

        attn_scores = q @ k.transpose(1, 0)
        output = attn_scores @ v
        output = output.transpose(1, 0).reshape(seq_len, 576)

        return output @ self.wo


In [80]:
text = "Anthropic is a company"

smollm2 = SmolLM2()
smollm2.next_token(text)


'<|endoftext|>'

In [ ]:
input_tokens = jnp.array(smollm2.tokenizer.encode("hello world!"))
embeddings = smollm2.model_weights["model.embed_tokens.weight"][input_tokens]

test_norm = RMSNorm(smollm2.model_weights["model.layers.0.input_layernorm.weight"])
test_transformer = GroupedQueryAttention(
                smollm2.model_weights["model.layers.0.self_attn.q_proj.weight"],
                smollm2.model_weights["model.layers.0.self_attn.k_proj.weight"],
                smollm2.model_weights["model.layers.0.self_attn.v_proj.weight"],
                smollm2.model_weights["model.layers.0.self_attn.o_proj.weight"],
            )
x = test_norm(embeddings)
test_transformer(x)

In [41]:
# wq = smollm2.model_weights["model.layers.0.self_attn.q_proj.weight"].reshape(9, 576 // 9, 576).transpose(2, 0, 1)
# wk = smollm2.model_weights["model.layers.0.self_attn.k_proj.weight"].reshape(3, 192 // 3, 576).transpose(2, 0, 1)
# wv = smollm2.model_weights["model.layers.0.self_attn.v_proj.weight"].reshape(3, 192 // 3, 576).transpose(2, 0, 1)

wq = smollm2.model_weights["model.layers.0.self_attn.q_proj.weight"]
wk = smollm2.model_weights["model.layers.0.self_attn.k_proj.weight"].repeat(3, axis=0)
wv = smollm2.model_weights["model.layers.0.self_attn.v_proj.weight"].repeat(3, axis=0)

q, k, v = x @ wq, x @ wk, x @ wv

scores =


In [62]:
smollm2_hf = AutoModelForCausalLM.from_pretrained("unsloth/SmolLM2-135M-Instruct", cache_dir="./data/hf")

config.json:   0%|          | 0.00/941 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

In [74]:
smollm2_hf.model

LlamaModel(
  (embed_tokens): Embedding(49153, 576, padding_idx=49152)
  (layers): ModuleList(
    (0-29): 30 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=False)
        (k_proj): Linear(in_features=576, out_features=192, bias=False)
        (v_proj): Linear(in_features=576, out_features=192, bias=False)
        (o_proj): Linear(in_features=576, out_features=576, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
        (up_proj): Linear(in_features=576, out_features=1536, bias=False)
        (down_proj): Linear(in_features=1536, out_features=576, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((576,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)

In [42]:
for i in smollm2.model_weights.keys():
    print(i)

model.embed_tokens.weight
model.layers.0.input_layernorm.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.post_attention_layernorm.weight
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.o_proj.weight
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.v_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.post_attention_layernorm.weight
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.o_proj.weight
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.v_proj.weight
model.layers.10.input_layernorm.weight
model.layers.10.mlp.down_proj.weight
model.layers.10.mlp.gate_proj.weight
model.layers.10.mlp.up_proj.weight
model.layers.10.post_attention_layernorm.weight
model.layers.10.self_attn.k_proj.weight
model.layers.10.self_attn.o_proj.weight
mode